# **File 3: api.py**

This is the script you keep running 24/7. It catches the normalized data from your ESP32, feeds it to the trained AI, and returns the physical hardware commands.

In [1]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from stable_baselines3 import PPO
import numpy as np
import math

app = FastAPI(title="Eco-Twin AI Brain API")

# Load the PyTorch model you generated in File 2
try:
    model = PPO.load("ppo_predictive_microgrid")
    print("AI Model loaded successfully.")
except Exception as e:
    print("Warning: Could not load model. Did you run train_ai.py first?")

# Define the exact JSON structure your teammate must send
class SensorData(BaseModel):
    battery_soc: float
    solar_forecast: list[float]  # Must be exactly 4 numbers (Now, +1, +2, +3)
    campus_demand: float
    hour_of_day: int

@app.post("/predict_action")
def get_ai_decision(data: SensorData):
    # Safety Check
    if len(data.solar_forecast) != 4:
        raise HTTPException(status_code=400, detail="solar_forecast must contain exactly 4 values.")

    # Time Encoding
    hour_sin = math.sin(2 * math.pi * data.hour_of_day / 24.0)
    hour_cos = math.cos(2 * math.pi * data.hour_of_day / 24.0)
    
    # Construct the PyTorch State Tensor
    obs = np.array([
        data.battery_soc, 
        data.solar_forecast[0], # Now
        data.solar_forecast[1], # +1hr
        data.solar_forecast[2], # +2hr
        data.solar_forecast[3], # +3hr
        data.campus_demand, 
        hour_sin, 
        hour_cos
    ], dtype=np.float32)
    
    # AI predicts the optimal action
    action, _ = model.predict(obs, deterministic=True)
    ai_command = float(action[0]) 
    
    # Translate to human/hardware readable text
    if ai_command < -0.1:
        status = "DISCHARGE"
    elif ai_command > 0.1:
        status = "CHARGE"
    else:
        status = "IDLE"
        
    return {
        "raw_action_value": ai_command,
        "hardware_command": status,
        "power_target_kw": round(abs(ai_command) * 5.0, 2), # Assuming 5kW max
        "message": f"AI commands battery to {status}."
    }